# Molecular Docking with Smina
## Standardized Virtual Screening of Trypsin (2ZQ2)

**Reference ligand:** `13U`  
**Screening library:** 117 candidate ligands  
**Validated docking set:** 111 ligands  
**Documented exceptions:** 6 ligands

---

> ### ⚠️ Authoritative workflow
> This notebook is **educational and analysis-focused**. It does **not** contain an independent production docking implementation.
>
> The authoritative standardized screening workflow is:
>
> `configs/docking_config.yml` → `scripts/run_standardized_docking.py` → `docking/results/standardized_affinities.csv`
>
> Production docking parameters are intentionally not duplicated here.

## What this notebook teaches

1. How the standardized docking configuration is represented.
2. How the 117-candidate library becomes the 111-ligand validated screening set.
3. How Smina docking outputs are organized and interpreted.
4. How to inspect and rank the **current authoritative** docking results.
5. Where downstream ProLIF and rule-based ADMET analyses fit into the workflow.

The notebook reads existing validated artifacts instead of silently creating a second docking pipeline.

## 1️⃣ Project setup

In [ ]:
import os
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "docking_config.yml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing configs/docking_config.yml."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

## 2️⃣ Load the authoritative docking configuration

In [ ]:
from src.docking_config import load_docking_config

config = load_docking_config(PROJECT_ROOT / "configs" / "docking_config.yml")

print("Target:", config.target_pdb)
print("Reference ligand:", config.reference_ligand)
print("Docking engine:", config.smina_executable)
print("Docking center (Å):", config.center)
print("Box size (Å):", config.box_size)
print("Exhaustiveness:", config.exhaustiveness)
print("Number of modes:", config.num_modes)
print("Seed:", config.seed)
print("Timeout (s):", config.timeout_seconds)
print("Expected validated ligands:", config.expected_ligands)
print("Receptor:", config.receptor)
print("Ligand directory:", config.ligand_dir)
print("Output directory:", config.output_dir)
print("Report:", config.report)

### Why the configuration is loaded instead of copied

The docking box and Smina settings are project-level protocol decisions. Keeping them in one validated YAML file prevents the notebook, scripts, and documentation from silently using different values.

## 3️⃣ Verify the prepared inputs

In [ ]:
protein_pdb = PROJECT_ROOT / "pdb" / "protein_h.pdb"
receptor_pdbqt = config.receptor
ligand_sdfs = sorted((PROJECT_ROOT / "ligands").glob("*.sdf"))
prepared_pdbqts = {path.stem for path in config.ligand_dir.glob("*.pdbqt")}

print(f"Hydrogenated protein exists: {protein_pdb.is_file()}")
print(f"Prepared receptor exists:     {receptor_pdbqt.is_file()}")
print(f"Candidate SDF files:          {len(ligand_sdfs)}")

protein_atom_count = 0
if protein_pdb.is_file():
    with protein_pdb.open() as handle:
        protein_atom_count = sum(
            line.startswith(("ATOM", "HETATM")) for line in handle
        )

print(f"Protein PDB atom records:     {protein_atom_count:,}")
print(f"Prepared ligand PDBQTs:        {len(prepared_pdbqts)}")

assert protein_pdb.is_file(), f"Missing prepared protein: {protein_pdb}"
assert receptor_pdbqt.is_file(), f"Missing receptor PDBQT: {receptor_pdbqt}"
assert len(ligand_sdfs) == 117, f"Expected 117 SDF files, found {len(ligand_sdfs)}"

The hydrogenated protein was prepared upstream in **Notebook 13**. The receptor PDBQT and ligand PDBQTs are treated here as inputs to the standardized docking stage; this notebook does not recreate them with an alternative method.

## 4️⃣ Understand the validated screening set

In [ ]:
manifest = pd.read_csv(config.manifest)

candidate_ids = {path.stem for path in ligand_sdfs}
validated_ids = set(manifest["ligand_id"].astype(str))

unresolved_ids = sorted(candidate_ids - validated_ids)
extra_validated_ids = sorted(validated_ids - candidate_ids)

print(f"Candidate ligands:   {len(candidate_ids)}")
print(f"Validated ligands:   {len(validated_ids)}")
print(f"Unresolved:           {len(unresolved_ids)}")
print("Unresolved IDs:", ", ".join(unresolved_ids) if unresolved_ids else "none")

assert len(candidate_ids) == 117
assert len(validated_ids) == config.expected_ligands
assert len(unresolved_ids) == 6
assert not extra_validated_ids
assert validated_ids.issubset(prepared_pdbqts)

### Current partition

`117 candidates = 111 validated + 6 documented exceptions`

The six unresolved ligands are intentionally outside the authoritative docking report. They are not silently counted as successful docking results.

## 5️⃣ Production docking is an explicit external step

The production run is executed from the repository root with:

```bash
python -m scripts.run_standardized_docking
```

That command reads the same configuration loaded above and writes the authoritative report and standardized pose files.

**This notebook does not call that command automatically.** Running a 111-ligand production screening job from a teaching notebook would create an unnecessary second execution path and make provenance harder to audit.

## 6️⃣ Load the authoritative standardized results

In [ ]:
from src.docking import get_top_hits, parse_smina_output

results = pd.read_csv(config.report)

required_columns = {"ligand_id", "affinity", "status", "message"}
missing_columns = required_columns - set(results.columns)

print(f"Reported ligands: {len(results)}")
print(f"Statuses: {results['status'].value_counts().to_dict()}")

assert not missing_columns, f"Missing report columns: {sorted(missing_columns)}"
assert len(results) == config.expected_ligands
assert results["ligand_id"].is_unique
assert set(results["status"]) == {"ok"}
assert results["affinity"].notna().all()

### Docking score interpretation

Smina reports a **computational docking score** in kcal/mol. Within this screening protocol, more negative scores rank ahead of less negative scores.

These values are ranking evidence from the docking model; they are **not experimental binding affinities**.

In [ ]:
top5 = get_top_hits(str(config.report), n=5)

display(
    top5[["ligand_id", "affinity", "status"]]
    .reset_index(drop=True)
    .rename(
        columns={
            "ligand_id": "Ligand",
            "affinity": "Docking score",
            "status": "Status",
        }
    )
)

In [ ]:
plt.figure(figsize=(8, 4.5))
top10 = results.sort_values("affinity").head(10).copy()
plt.bar(top10["ligand_id"], top10["affinity"])
plt.xlabel("Ligand")
plt.ylabel("Docking score (kcal/mol)")
plt.title("Top 10 standardized docking scores")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7️⃣ Inspect a standardized Smina pose

A standardized output file contains the pose together with the machine-readable `REMARK minimizedAffinity` record used by the reusable parser.

In [ ]:
example_ligand = str(top5.iloc[0]["ligand_id"])
example_pose = config.output_dir / f"{example_ligand}_out.pdbqt"
parsed_scores = parse_smina_output(str(example_pose))

print(f"Ligand: {example_ligand}")
print(f"Pose file: {example_pose}")
print(f"Parsed minimizedAffinity values: {parsed_scores}")

assert example_pose.is_file()
assert parsed_scores
assert parsed_scores[0] == top5.iloc[0]["affinity"]

## 8️⃣ Downstream analyses

The current workflow continues from the authoritative docking report:

```text
standardized_affinities.csv
        │
        ├──► prepare_top5.py
        │       └──► current Top 5 structures
        │
        ├──► prolif_analysis.py
        │       └──► docking/results/prolif_summary.csv
        │
        └──► admet_top5.py
                └──► docking/results/admet/
```

The final combined table is produced by `scripts/merge_final_results.py`.

The important architectural rule is that downstream analyses consume the **current standardized docking results**, rather than the older `affinities_full.csv` / legacy pose directories.

## 🧾 Current standardized Top 5

| Rank | Ligand | Docking score (kcal/mol) |
|---:|---|---:|
| 1 | **R11** | **−9.7876** |
| 2 | **13U** | **−9.5111** |
| 3 | **BAH** | **−9.4138** |
| 4 | **12U** | **−9.3740** |
| 5 | **T87** | **−9.2705** |

These values are read from the current authoritative report when the notebook runs; the table above documents the validated current state for teaching purposes.

## ✅ Notebook summary

This notebook now has a single responsibility:

> **Teach and inspect the standardized docking stage without creating a competing production pipeline.**

The authoritative chain is:

```text
117 candidate ligands
        ↓
111 validated ligands + 6 documented exceptions
        ↓
configs/docking_config.yml
        ↓
scripts/run_standardized_docking.py
        ↓
docking/results/standardized_affinities.csv
        ↓
Current Top 5
        ↓
ProLIF / rule-based ADMET / final merge
```

Historical docking artifacts and the separate solved notebook are intentionally not used to define the current ranking.